# 01 (DRAC-local variant) — reasoning_gym GRPO against a LOCAL in-job env server

This is a COPY of `01_reasoning_gym_sft_then_grpo.ipynb`, identical except `ENV_BASE_URL`
defaults to **`http://localhost:8000`** instead of the hosted Space.

**Why:** DRAC compute nodes block outbound WebSocket (`wss://`) to third-party Spaces, so the
hosted-env path times out. Instead, the SLURM runner `jobs/openenv-grpo-reasoning-gym-local.sh`
launches the env's own uvicorn server **in the same job** (`uvicorn reasoning_gym_env.server.app:app`)
and this notebook connects to it over **loopback** (`http://localhost:8000` -> `ws://localhost:8000/ws`),
which no firewall blocks. Everything else (SFT-warmup, GRPO, eval) is unchanged.

On a host with open egress (Colab, cloud GPU) set `ENV_BASE_URL` to a hosted Space and it behaves
like the original notebook.


# 01 · Reasoning-Gym — SFT warm-start → GRPO → evaluate the agent

This is the **full, explained pipeline** for training an agent with OpenEnv + the HF
ecosystem, faithful to the canonical HF tutorials (Sergio Paniego / Ben Burtenshaw):

1. **Collect** reward-labeled rollouts from a *teacher* model running in the env.
2. **Filter** to the successful ones (`reward == 1.0`) and **SFT** a student on them —
   a warm-start so GRPO does not begin from random tool-calling behavior.
3. **GRPO** (Group Relative Policy Optimization): the env returns a scalar reward per
   rollout, and GRPO ranks rollouts of the same prompt against each other. Value-free,
   so the env's reward is the *only* signal it needs.
4. **Evaluate** the trained agent on a **held-out seed** and report accuracy +
   format-compliance, base vs. trained.

> **Where does a "trace dataset" (e.g. Fable) fit?** It does **not** replace the env.
> OpenEnv's training data is **reward-labeled rollouts the environment generates**, tied
> to *this env's* tool surface (`answer`). Generic imitation traces are not graded by the
> env and do not exercise its tools, so they are not a substitute. A trace corpus could
> only serve as an optional SFT warm-start *if* its tool surface matched the env's —
> which Fable's (Claude-Code-style) does not. We therefore collect our own rollouts below.

> **Runtime targets — this notebook runs anywhere.** Colab (A100/L4/T4), a fresh
> local GPU box, or an HPC cluster via the SLURM runner in `jobs/`. It self-installs
> its dependencies, authenticates with `notebook_login()` (or `HF_TOKEN`), connects
> to an environment by **URL** (a variable you can repoint to your own Space), and
> auto-detects the GPU with a model-size fallback. There are **no hard-coded paths**.

<!-- @inject:run-recipes -->
## ▶ How to run this notebook off-DRAC (HF Jobs · Google Colab)

This notebook is portable — **no DRAC required**. Pick a lane. (Dependencies and the env
URL below are taken from this notebook's own cells.)

### A · HF Jobs — run the notebook on Hugging Face infra

`hf jobs run` is *docker-run for HF infra*: the compute happens on HF's cloud, you only
submit. The control plane is plain HTTPS, so this works even from networks that block the
hosted-env WebSocket (e.g. DRAC compute). Requires pre-paid credits on your HF account.

```bash
# one-time: hf auth login   (or rely on a local HF_TOKEN)
hf jobs run \
  --flavor a10g-small \
  --timeout 10800 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=http://localhost:8000 \
  python:3.12 \
  bash -c "pip install -q papermill && \
           pip install -q trl openenv "transformers>=5.3.0" trackio openai jmespath nest_asyncio datasets && \
           pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym && \
           papermill 01_reasoning_gym_sft_then_grpo_DRAC_local.ipynb out.ipynb"
```

Notes: `--secrets HF_TOKEN` forwards your local token (needed if the notebook pushes to the
Hub). `--flavor` options: `cpu-basic`, `t4-small`, `a10g-small`, `a100-large`, … (`hf jobs`
docs). Add `--detach` to background it; watch with `hf jobs logs <id>`.

### B · HF Jobs — full run with vLLM (faster rollouts)

Same as A but on a bigger GPU with vLLM enabled (`USE_VLLM=1`). vLLM works here because an
HF Job is **not** an IPython runtime (the reason it stays OFF in-notebook).

```bash
hf jobs run \
  --flavor a100-large \
  --timeout 21600 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e USE_VLLM=1 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=http://localhost:8000 \
  python:3.12 \
  bash -c "pip install -q papermill vllm && \
           pip install -q trl openenv "transformers>=5.3.0" trackio openai jmespath nest_asyncio datasets && \
           pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym && \
           papermill 01_reasoning_gym_sft_then_grpo_DRAC_local.ipynb out.ipynb"
```

### C · Google Colab — independent run

Open Colab → set runtime to **GPU** → in the first cell:

```python
# Colab cell 1 — auth + knobs, then run the rest of the notebook top-to-bottom.
import os
os.environ["SMOKE"] = "0"                 # full run (omit for a quick smoke)
os.environ["REPORT_TO"] = "trackio"       # live charts; or "none" for headless
os.environ["ENV_BASE_URL"] = "http://localhost:8000"
from huggingface_hub import notebook_login
notebook_login()                          # paste a token with write access
```

Then **Runtime → Run all**. Colab has open egress, so the hosted env Space works directly
(no in-job uvicorn needed). The install cell at the top of the notebook handles
dependencies.

For a longer run on an A100 Colab, set `os.environ["USE_VLLM"] = "1"` *before* the GRPO config cell — Colab's kernel tolerates vLLM colocate on a single A100.

---


## 0 · Install

`environment_factory` lives in current TRL and needs `transformers>=5.3.0`. The env
client ships from its Space. `trackio` gives live training charts; `openai` is only
needed if you choose the OpenAI teacher in the collect step (optional).

In [ ]:
%pip install -q trl openenv "transformers>=5.3.0" trackio openai jmespath nest_asyncio datasets
%pip install -q --no-deps git+https://huggingface.co/spaces/sergiopaniego/reasoning_gym

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
# --- Authenticate with the Hugging Face Hub (portable) -------------------------
# Prefers an already-set HF_TOKEN (CI / SLURM / Colab secret); falls back to the
# interactive widget. Never hard-codes a token.
import os

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"])
    print("Authenticated via HF_TOKEN.")
else:
    try:
        from huggingface_hub import notebook_login

        notebook_login()
    except Exception as exc:  # non-interactive without a token
        print(f"notebook_login unavailable ({exc}); set HF_TOKEN before running.")

In [ ]:
# Resolve your Hub username automatically — no hard-coded usernames downstream.
from huggingface_hub import whoami

try:
    HF_USERNAME = whoami()["name"]
    print(f"Hub user: {HF_USERNAME}")
except Exception as exc:
    HF_USERNAME = os.environ.get("HF_USERNAME", "")
    print(f"Could not resolve whoami() ({exc}); using HF_USERNAME={HF_USERNAME!r}.")

In [ ]:
# --- Auto-detect compute + pick a model that fits ------------------------------
# Larger model only when there is VRAM headroom; otherwise fall back. You can
# override by setting MODEL_NAME in the environment before launching.
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    DEVICE = "cuda"
else:
    vram_gb = 0.0
    DEVICE = "cpu"

DEFAULT_MODEL = "Qwen/Qwen3-1.7B" if vram_gb >= 24 else "Qwen/Qwen3-0.6B"
MODEL_NAME = os.environ.get("MODEL_NAME", DEFAULT_MODEL)
print(f"device={DEVICE}  vram={vram_gb:.1f} GB  ->  MODEL_NAME={MODEL_NAME}")

In [ ]:
# --- Run knobs (smoke vs. full) ------------------------------------------------
# SMOKE keeps everything tiny so the whole notebook runs in minutes on one GPU to
# prove the pipeline end-to-end. Set SMOKE=False (or env SMOKE=0) for a real run.
SMOKE = os.environ.get("SMOKE", "1") not in ("0", "false", "False")

ENV_BASE_URL = os.environ.get("ENV_BASE_URL", "http://localhost:8000")
DATASET_NAME = "chain_sum"
DATASET_CONFIG = {"min_terms": 2, "max_terms": 3, "min_digits": 2, "max_digits": 2}

N_EPISODES_COLLECT = 30 if SMOKE else 300     # teacher rollouts for SFT
GRPO_MAX_STEPS = 5 if SMOKE else 150          # GRPO optimisation steps
N_EVAL = 10 if SMOKE else 50                  # held-out eval problems
print(f"SMOKE={SMOKE}  collect={N_EPISODES_COLLECT}  grpo_steps={GRPO_MAX_STEPS}  eval={N_EVAL}")
print(f"env={ENV_BASE_URL}")

## 1 · (Optional) Deploy your own environment Space

Hosted tutorial Spaces are **low-concurrency** — fine for this notebook, a bottleneck
for a real run where the trainer spins up many parallel envs. For production, duplicate
the Space to your own account (`openenv push`, or the Hub "Duplicate this Space" button)
and set `ENV_BASE_URL` to it. Left as a documented step so the notebook stays runnable
out-of-the-box against the hosted default.

In [ ]:
# Uncomment to deploy your own copy (requires Docker locally OR use the Hub UI button):
#   !openenv push sergiopaniego/reasoning_gym --to {HF_USERNAME}/reasoning_gym
# then:  ENV_BASE_URL = f"https://{HF_USERNAME.replace('_','-')}-reasoning-gym.hf.space"
print("Using ENV_BASE_URL =", ENV_BASE_URL)

## 2 · Phase A — collect teacher rollouts (the "dataset")

We run a **teacher** model through the env with the OpenEnv *harness*, which records
each episode's messages and the env's reward. This is the real OpenEnv data-generation
step — `CollectRunner` drives the teacher, `RolloutSerializer` writes episodes to disk,
`push_to_hf_hub` publishes them as a dataset.

The teacher is a **parameter**. The tutorial uses OpenAI `gpt-5-mini`; for a fully-open,
no-API-key path you can point `create_llm_client` at any OpenAI-compatible endpoint
(e.g. a TGI/vLLM server). Set `TEACHER_PROVIDER` / `TEACHER_MODEL` to choose.

In [ ]:
SYSTEM_PROMPT = """You are a careful arithmetic assistant.

You will be given a chain of integer additions. Compute the result and submit it as a single number.

Rules:
1. Read the question carefully.
2. Use the tool `answer` exactly once with your final number.
3. The answer must be a single integer with no units or explanation.
"""

In [ ]:
# Teacher is configurable. Tutorial-faithful default is OpenAI gpt-5-mini, BUT a teacher
# requires a reachable endpoint: an OpenAI key, OR a self-hosted OpenAI-compatible URL.
# If neither is available we cannot collect — fail LOUD and early with a clear message
# rather than deep inside CollectRunner. (Set TEACHER_BASE_URL to a TGI/vLLM server for a
# fully-open, no-key path.)
TEACHER_PROVIDER = os.environ.get("TEACHER_PROVIDER", "openai")
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "gpt-5-mini")
TEACHER_BASE_URL = os.environ.get("TEACHER_BASE_URL")  # set for a self-hosted endpoint

_has_openai_key = bool(os.environ.get("OPENAI_API_KEY"))
TEACHER_AVAILABLE = bool(TEACHER_BASE_URL) or (TEACHER_PROVIDER == "openai" and _has_openai_key)
if not TEACHER_AVAILABLE:
    print(
        "[skip-collect] No teacher endpoint available "
        "(set OPENAI_API_KEY, or TEACHER_BASE_URL for an OpenAI-compatible server).\n"
        "Phase A (collect+SFT) will be SKIPPED; GRPO will cold-start from the base model.\n"
        "This keeps the notebook runnable end-to-end with zero external keys."
    )

In [ ]:
# Collect only if a teacher is reachable. Otherwise we skip straight to GRPO (cold-start),
# so the notebook always runs top-to-bottom.
RUN_SFT = TEACHER_AVAILABLE
correct = []  # populated by Phase A when it runs

if TEACHER_AVAILABLE:
    from openenv.core.harness import HarnessRunLimits, MCPHarnessAdapter
    from openenv.core.harness.collect import (
        CollectRunner,
        RolloutSerializer,
        build_model_step,
        push_to_hf_hub,
    )
    from openenv.core.llm_client import create_llm_client
    from reasoning_gym_env.client import ReasoningGymEnv
    from reasoning_gym_env.harness import ReasoningGymSessionFactory

    _client_kwargs = {"provider": TEACHER_PROVIDER, "model": TEACHER_MODEL, "max_tokens": 1024}
    if TEACHER_PROVIDER == "openai" and _has_openai_key:
        _client_kwargs["api_key"] = os.environ["OPENAI_API_KEY"]
    if TEACHER_BASE_URL:
        _client_kwargs["base_url"] = TEACHER_BASE_URL

    llm_client = create_llm_client(**_client_kwargs)
    model_step = build_model_step(llm_client, system_prompt=SYSTEM_PROMPT)

    factory = ReasoningGymSessionFactory(
        lambda: ReasoningGymEnv(base_url=ENV_BASE_URL),
        dataset_name=DATASET_NAME,
        dataset_config=DATASET_CONFIG,
    )
    serializer = RolloutSerializer("./rollouts")
    runner = CollectRunner(
        session_factory=factory,
        harness_adapter=MCPHarnessAdapter(),
        serializer=serializer,
        limits=HarnessRunLimits(max_turns=9),
    )
    result = runner.run(model_step=model_step, num_episodes=N_EPISODES_COLLECT)
    print(
        f"collected={result.num_collected} dropped={result.num_dropped} "
        f"avg_reward={result.avg_reward:.3f} success_rate={result.success_rate:.0%}"
    )

In [ ]:
# Publish the rollouts to the Hub only when authenticated (optional, for sharing/reuse).
# The SFT step reads from the LOCAL ./rollouts dir, so a Hub push is never required to run.
if TEACHER_AVAILABLE and HF_USERNAME:
    try:
        url = push_to_hf_hub(output_dir="./rollouts", repo_id=f"{HF_USERNAME}/chain-sum-rollouts")
        print(f"Rollouts dataset: {url}")
    except Exception as exc:
        print(f"[warn] Hub push skipped ({exc}); continuing from local ./rollouts.")

## 3 · Filter + format the rollouts for SFT

Keep only episodes where the teacher was **correct** (`reward == 1.0`), and rewrite the
assistant tool call into the `<tool_call>{json}</tool_call>` text form the student learns
to emit. We strip the env's `tool` responses — SFT only supervises the assistant turn.
We read rollouts from the **local** `./rollouts` directory (no Hub round-trip needed).

In [ ]:
import json

from datasets import load_dataset

if RUN_SFT:
    # Load from local disk — portable, no Hub auth required.
    ds = load_dataset("json", data_files="./rollouts/*.jsonl", split="train")
    raw_rollouts = list(ds)
    print(f"loaded {len(raw_rollouts)} episodes from ./rollouts")
else:
    raw_rollouts = []
    print("Phase A skipped (no teacher) — no rollouts to format.")


def to_chat_messages(record):
    converted = []
    for msg in record["messages"]:
        if msg["role"] == "tool":
            continue  # SFT supervises only the assistant turn
        if msg["role"] == "assistant" and msg.get("tool_calls"):
            tc = msg["tool_calls"][0]
            args = json.loads(tc["function"]["arguments"])
            tool_call_text = (
                "<tool_call>\n"
                + json.dumps({"name": "answer", "arguments": {"answer": args.get("answer", "")}})
                + "\n</tool_call>"
            )
            converted.append({"role": "assistant", "content": tool_call_text})
        else:
            converted.append(msg)
    return {"messages": converted, "reward": record["reward"]}


import re as _re


def _slug(s: str) -> str:
    # HF/trackio Space ids prefer hyphen-lowercase; underscores get normalised in the
    # Space subdomain, so build a clean slug up front (matches the canonical naming).
    return _re.sub(r"-+", "-", _re.sub(r"[^a-z0-9]+", "-", s.lower())).strip("-")


_RUN_TAG = _slug(f"reasoning-gym-{DATASET_NAME}-{MODEL_NAME.split('/')[-1]}")
SFT_OUT = f"{_RUN_TAG}-sft"

rollouts = [to_chat_messages(r) for r in raw_rollouts]
correct = [r for r in rollouts if r["reward"] == 1.0]
if RUN_SFT:
    print(f"correct: {len(correct)} / {len(rollouts)} ({len(correct) / max(len(rollouts),1):.1%})")
    if not correct:
        # No usable teacher rollouts -> skip SFT rather than abort; GRPO cold-starts.
        RUN_SFT = False
        print("[skip-sft] 0 correct rollouts; GRPO will cold-start from the base model.")

## 4 · Size the sequence length from the data

When SFT runs, rather than guess `max_length` we measure it: tokenize the kept rollouts
and set the cap at the 99th percentile + a small margin. Data-driven, model-agnostic.

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_SEQ_LEN = 1024  # default if SFT is skipped
if RUN_SFT and correct:
    lengths = []
    for row in correct:
        text = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
        lengths.append(len(tokenizer.encode(text)))
    lengths = np.array(lengths)
    MAX_SEQ_LEN = int(np.percentile(lengths, 99)) + 16
    print(f"p50={np.percentile(lengths,50):.0f} p95={np.percentile(lengths,95):.0f} "
          f"p99={np.percentile(lengths,99):.0f} max={lengths.max()}  -> MAX_SEQ_LEN={MAX_SEQ_LEN}")
else:
    print(f"SFT skipped; MAX_SEQ_LEN default = {MAX_SEQ_LEN}")

## 5 · Phase A — SFT warm-start

`assistant_only_loss=True` masks the loss to the assistant tokens, so the student learns
*to produce the tool call*, not to parrot the prompt. The result is a checkpoint that
already calls the `answer` tool in the right format — a far better GRPO starting point
than the base model. **Runs only when Phase A collected usable rollouts**; otherwise GRPO
cold-starts from the base model and the notebook continues.

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

if RUN_SFT and correct:
    sft_dataset = Dataset.from_list([{"messages": r["messages"]} for r in correct])
    sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    sft_config = SFTConfig(
        output_dir=SFT_OUT,
        hub_model_id=f"{HF_USERNAME}/{SFT_OUT}-model" if HF_USERNAME else f"{SFT_OUT}-model",
        max_length=MAX_SEQ_LEN,
        num_train_epochs=1 if SMOKE else 3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        assistant_only_loss=True,
        push_to_hub=not SMOKE,
    )
    sft_trainer = SFTTrainer(
        model=sft_model,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
        args=sft_config,
    )
    sft_trainer.train()
    sft_trainer.save_model(SFT_OUT)
    if not SMOKE:
        sft_trainer.push_to_hub(commit_message="SFT warm-up on reasoning_gym chain_sum")
    print(f"SFT checkpoint -> {SFT_OUT}")
else:
    print("SFT skipped — GRPO will cold-start from", MODEL_NAME)

## 6 · Phase B — GRPO

Now the RL phase. The pieces, all verified from the canonical walkthrough:

- **An env wrapper class.** Its public, documented methods become the agent's *tools*.
  Here `answer(...)` is the single tool; `reset()` pulls the next question. The class
  tracks `self.reward` / `self.done` so the trainer can read the outcome.
- **A reward function** that just reads `env.reward` off each rollout's env instance.
- **A dummy prompt dataset** whose only job is to set the episode count — the *real*
  prompts come from `env.reset()` inside the factory.
- **`environment_factory=<class>`** on `GRPOTrainer`: the trainer creates one env per
  rollout, generates the completion, parses the tool call, steps the env, and collects
  the reward — the entire loop from notebook 00, automated.

In [ ]:
import random

from reasoning_gym_env import ReasoningGymAction, ReasoningGymEnv


class ReasoningGymTrainEnv:
    """One rollout = one question -> one `answer` tool call -> done."""

    DATASET_SIZE = 1000

    def __init__(self):
        self.client = ReasoningGymEnv(base_url=ENV_BASE_URL).sync()
        self._dataset_seed = random.randint(0, 2**31 - 1)  # per-instance: parallel envs diverge
        self._initialized = False
        self.reward = 0.0
        self.done = False

    def reset(self, **kwargs) -> str:
        if not self._initialized:
            result = self.client.reset(
                dataset_name=DATASET_NAME,
                dataset_config=DATASET_CONFIG,
                seed=self._dataset_seed,
                size=self.DATASET_SIZE,
            )
            self._initialized = True
        else:
            result = self.client.reset()  # next question; re-sending config would rewind to 0
        self.reward = 0.0
        self.done = False
        return result.observation.question

    def answer(self, answer: str) -> str:
        """Submit the final answer for the current question.

        Args:
            answer: The agent's answer (parsed as a number server-side).

        Returns:
            Feedback string with the score and the correct answer.
        """
        if self.done:
            raise ValueError("Episode is already finished.")
        result = self.client.step(ReasoningGymAction(answer=str(answer)))
        self.reward = float(result.observation.score or 0.0)
        self.done = True
        return f"score={self.reward} correct={result.observation.correct_answer}"


def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]

In [ ]:
from datasets import Dataset

# Dummy prompts: count == number of rollout episodes. Real prompts come from reset().
N_PROMPTS = 50 if SMOKE else 1000
grpo_dataset = Dataset.from_dict(
    {"prompt": [[{"role": "user", "content": SYSTEM_PROMPT}] for _ in range(N_PROMPTS)]}
)

In [ ]:
from trl import GRPOConfig

GRPO_OUT = f"{_RUN_TAG}-grpo"

# Logging backend is a knob: "trackio" (default, live charts) or "none" for a fully
# offline/headless run with no Space setup. trackio_space_id is only set when using trackio.
REPORT_TO = os.environ.get("REPORT_TO", "trackio")
_report_kwargs = {"report_to": REPORT_TO}
if REPORT_TO == "trackio":
    _report_kwargs["trackio_space_id"] = GRPO_OUT

grpo_config = GRPOConfig(
    num_train_epochs=1,
    max_steps=GRPO_MAX_STEPS,
    learning_rate=1e-6,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=1,
    warmup_steps=min(10, GRPO_MAX_STEPS),
    optim="adamw_torch",
    max_grad_norm=1.0,
    num_generations=2,
    max_completion_length=256,
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs={"enable_thinking": False},
    output_dir=GRPO_OUT,
    # Push weights to a repo DISTINCT from the Trackio Space id (=GRPO_OUT),
    # otherwise push_to_hub sees the Trackio-owned repo and skips as 'no files
    # modified'. A separate -model repo gets the actual checkpoint commit.
    hub_model_id=f"{HF_USERNAME}/{GRPO_OUT}-model" if HF_USERNAME else f"{GRPO_OUT}-model",
    logging_steps=1 if SMOKE else 10,
    gradient_checkpointing=True,
    save_strategy="no",
    push_to_hub=not SMOKE,
    **_report_kwargs,
    # vLLM left OFF in-notebook: its init breaks under IPython. Enable in the SLURM/script
    # path with use_vllm=True, vllm_mode="colocate".
)

In [ ]:
# @inject:vllm-optional
# --- OPTIONAL: vLLM-accelerated GRPO rollouts (5-10x faster generation) ---------
# OFF by default: vLLM's init breaks under IPython/Jupyter, so leave USE_VLLM=0 for an
# in-notebook run. Enable it (USE_VLLM=1) ONLY in a non-IPython context: an HF Job, a
# SLURM/script run, or a fresh Colab runtime executed as a script. Colocate mode shares
# the single training GPU (right for Colab / one-GPU HF-Job flavors).
#   Verified TRL v1.7.0 params: use_vllm, vllm_mode="colocate"|"server",
#   vllm_gpu_memory_utilization. (server mode is multi-GPU; not used here.)
import os

USE_VLLM = os.environ.get("USE_VLLM", "0") not in ("0", "false", "False")
if USE_VLLM:
    grpo_config.use_vllm = True
    grpo_config.vllm_mode = "colocate"
    # Leave room for the training copy of the model in colocate mode; tune per GPU/model.
    grpo_config.vllm_gpu_memory_utilization = float(
        os.environ.get("VLLM_GPU_MEM_UTIL", "0.3")
    )
    print(
        f"[vLLM] enabled: mode=colocate gpu_mem_util={grpo_config.vllm_gpu_memory_utilization} "
        "(requires a non-IPython runtime + `pip install vllm`)"
    )
else:
    print("[vLLM] disabled (USE_VLLM=0). In-notebook generation uses HF generate().")


In [ ]:
from trl import GRPOTrainer

# Warm-start GRPO from the SFT checkpoint when available; else from base.
import os as _os

# Warm-start from THIS run's SFT checkpoint only. Guard on RUN_SFT so a stale SFT dir
# left in the cwd by a PRIOR run does not silently warm-start a cold-start run.
GRPO_INIT = SFT_OUT if (RUN_SFT and _os.path.isdir(SFT_OUT)) else MODEL_NAME
print(f"GRPO initialising from: {GRPO_INIT}")

grpo_trainer = GRPOTrainer(
    model=GRPO_INIT,
    reward_funcs=reward_func,
    train_dataset=grpo_dataset,
    args=grpo_config,
    environment_factory=ReasoningGymTrainEnv,
)
grpo_trainer.train()
grpo_trainer.save_model(GRPO_OUT)
if not SMOKE:
    grpo_trainer.push_to_hub(commit_message="GRPO fine-tune on reasoning_gym chain_sum")

## 7 · Training signal — reward delta

The quickest sanity check: did mean reward rise over training? We read it from the
trainer's log history (first-5 vs last-5 logged steps).

In [ ]:
import statistics

rewards = [log["reward"] for log in grpo_trainer.state.log_history if "reward" in log]
if len(rewards) < 5:
    print(f"Only {len(rewards)} reward logs — increase max_steps / lower logging_steps for a clean delta.")
else:
    initial, final = statistics.mean(rewards[:5]), statistics.mean(rewards[-5:])
    print(f"initial reward (first5): {initial:.2%}")
    print(f"final   reward (last5):  {final:.2%}")
    print(f"delta:                   {(final - initial) * 100:+.2f} pp")

## 8 · Test the trained agent against held-out problems

Run both the base model and the GRPO checkpoint on a held-out `seed` (problems not seen
in training) and report two numbers, base vs. trained:

- **accuracy** — fraction of held-out problems the env scores correct;
- **format compliance** — fraction where the model emitted a parseable `answer`.

This is the reference `evaluate_model` harness from the HF SFT-warmup tutorial.

In [ ]:
import re

from transformers import pipeline
from reasoning_gym_env.client import ReasoningGymEnv
from reasoning_gym_env.models import ReasoningGymAction


async def evaluate_model(model_name_or_path, n_eval=N_EVAL, seed=999):
    """Held-out accuracy + format-compliance for one model (reference strategy).

    Single-turn: render [system, user], generate once, extract the answer, step the env
    once, read the env score. `format_compliance` = fraction emitting a parseable
    `"answer": <int>` (the trained tool-call surface). Higher seed => unseen problems.
    """
    gen = pipeline(
        "text-generation",
        model=model_name_or_path,
        tokenizer=model_name_or_path,
        device_map="auto",
        dtype="auto",
    )
    gen.model.generation_config.max_length = None
    tok = AutoTokenizer.from_pretrained(model_name_or_path)
    eval_env = ReasoningGymEnv(base_url=ENV_BASE_URL)

    obs = await eval_env.reset(
        dataset_name=DATASET_NAME, dataset_config=DATASET_CONFIG, seed=seed, size=n_eval
    )

    rewards, format_hits = [], 0
    for i in range(n_eval):
        if i:
            obs = await eval_env.reset()
        question = obs.observation.question
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        completion = gen(prompt, max_new_tokens=256)[0]["generated_text"][len(prompt):]

        # Reference extraction: `"answer": <int>` sets format compliance; else last integer.
        # Signed ints kept (our param): chain_sum results can be negative.
        m = re.search(r'"answer"\s*:\s*"?(-?\d+)"?', completion)
        if m:
            format_hits += 1
            answer = m.group(1)
        else:
            nums = re.findall(r"(-?\d+)", completion)
            answer = nums[-1] if nums else "0"

        res = await eval_env.step(ReasoningGymAction(answer=answer))
        rewards.append(float(res.observation.score or 0.0))

    await eval_env.close()
    del gen
    return {"accuracy": sum(rewards) / len(rewards), "format_compliance": format_hits / n_eval}


In [ ]:
# Compare base vs. the GRPO-trained checkpoint on held-out problems.
base_metrics = await evaluate_model(MODEL_NAME)
trained_metrics = await evaluate_model(GRPO_OUT)

print(f"\n{'Metric':<22}{'Base':>10}{'Trained':>10}{'Delta':>10}")
print("-" * 52)
for key, label in [("format_compliance", "Format compliance"), ("accuracy", "Accuracy")]:
    b, t = base_metrics[key], trained_metrics[key]
    print(f"{label:<22}{b:>10.1%}{t:>10.1%}{(t - b) * 100:>+9.1f}pp")


## Recap

You ran the complete OpenEnv agent-training loop: **collect → filter → SFT warm-start →
GRPO → held-out evaluation**, all against a live environment, all portable. The trained
agent's quality is measured the only way that counts — accuracy on problems it never saw.

- For a real run: set `SMOKE=False`, deploy your own env Space, and enable vLLM in the
  SLURM path (`jobs/openenv-grpo-reasoning-gym.sh`).
- **Next:** `02_wordle_agentic_grpo.ipynb` applies the same shape to a genuinely
  *multi-turn* agentic environment (6 guesses per episode with stateful feedback).